# Week 12 Student Lab: Image Classification & Land Cover Mapping — ARIA v8.0

## 學生實作筆記本

延續 W8–W10 的 STAC API 工作流程，本週進入影像分類與土地覆蓋製圖。

- **Lab 1:** K-means 非監督式分類
- **Lab 2:** Random Forest 監督式分類
- **應用情境:** 2024 花蓮地震後崩塌地偵測與面積估算

---

### 使用說明

- 標記為 **COMPLETE** 的儲存格已預先填好，直接執行即可。
- 標記為 `# ── YOUR TURN ──` 的區塊需要你完成程式碼。
- 每個 YOUR TURN 區塊都附有 `# Hint:` 提示，請仔細閱讀。
- 完成後依序執行所有儲存格，觀察輸出結果。

In [ ]:
# =============================================================================
# [S1] 環境設定 + STAC API 連線 + 研究區域定義 【COMPLETE — 直接執行】
# =============================================================================

# --- 基本套件 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# --- 中文字型設定（跨平台：Colab / Windows / macOS 皆適用）---
import matplotlib.font_manager as fm
try:
    import subprocess
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"],
                   capture_output=True)
    fm._load_fontmanager(try_read_cache=False)
except FileNotFoundError:
    pass  # 非 Linux（Windows / macOS），跳過 apt-get
_cjk_candidates = [
    "Microsoft JhengHei",      # Windows 正黑體
    "PingFang TC",             # macOS 蘋方
    "Noto Sans CJK TC",       # Colab / Linux
    "Heiti TC",                # macOS 黑體
    "WenQuanYi Micro Hei",    # Linux
    "Droid Sans Fallback",    # Android / 舊 Linux
]
_available = {f.name for f in fm.fontManager.ttflist}
_cjk_found = [f for f in _cjk_candidates if f in _available]
if _cjk_found:
    plt.rcParams["font.sans-serif"] = _cjk_found + ["DejaVu Sans"]
    print(f"中文字型: {_cjk_found[0]}")
else:
    print("⚠️ 未找到中文字型，中文標題可能顯示為方框")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# --- STAC / 遙測資料載入 ---
import pystac_client
import planetary_computer
import stackstac

# --- 機器學習 ---
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    cohen_kappa_score,
)
from sklearn.model_selection import train_test_split

# --- 影像後處理 ---
from scipy.ndimage import median_filter

# --- GIS / 向量處理（ROI 多邊形 → 像素遮罩）---
import geopandas as gpd
from shapely.geometry import Polygon
from rasterio.features import rasterize
from rasterio.transform import Affine
import pyproj
import zipfile, xml.etree.ElementTree as ET  # KMZ 解析用

print("所有套件載入成功！")

# --- 連線 Planetary Computer STAC API ---
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("STAC API 連線成功:", catalog.title)

# --- 研究區域：花蓮市周邊（2024 地震災區）---
# 東邊界擴大至 121.6、北邊界至 23.77，以涵蓋近海水體樣本區
HUALIEN_BBOX = [121.2574, 23.6546, 121.6, 23.77]

print(f"研究區域 BBOX: {HUALIEN_BBOX}")
print(f"經度範圍: {HUALIEN_BBOX[0]:.4f} ~ {HUALIEN_BBOX[2]:.4f}")
print(f"緯度範圍: {HUALIEN_BBOX[1]:.4f} ~ {HUALIEN_BBOX[3]:.4f}")

In [ ]:
# =============================================================================
# [S2] 搜尋 Sentinel-2 震後影像 【COMPLETE — 直接執行】
# =============================================================================

# 漸進式搜尋：先嚴格，找不到就自動放寬
# （花蓮 4-5 月是梅雨季，雲量高，需要彈性搜尋策略）
search_configs = [
    ("2024-04-15/2024-05-31", 20, "Phase 1: 震後 2 個月，雲量 < 20%"),
    ("2024-04-03/2024-08-31", 30, "Phase 2: 震後 5 個月，雲量 < 30%"),
    ("2024-04-03/2024-12-31", 50, "Phase 3: 震後全年，雲量 < 50%"),
    ("2024-01-01/2024-12-31", 70, "Phase 4: 2024 全年，雲量 < 70%"),
]

items = None
for dt_range, max_cc, desc in search_configs:
    print(f"嘗試 {desc}...")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=HUALIEN_BBOX,
        datetime=dt_range,
        query={"eo:cloud_cover": {"lt": max_cc}},
    )
    items = search.item_collection()
    print(f"  → 找到 {len(items)} 景影像")
    if len(items) > 0:
        break

if len(items) == 0:
    raise RuntimeError(
        "即使放寬條件仍未找到影像。請檢查網路連線或 Planetary Computer 服務狀態。"
    )

# 依雲量排序，選取最佳（最低雲量）影像
items_sorted = sorted(items, key=lambda x: x.properties["eo:cloud_cover"])
best_item = items_sorted[0]

print("\n=== 選取的最佳影像 ===")
print(f"  影像 ID    : {best_item.id}")
print(f"  拍攝日期  : {best_item.datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  雲量      : {best_item.properties['eo:cloud_cover']:.1f}%")
print(f"  平台      : {best_item.properties.get('platform', 'N/A')}")
print(f"  可用波段  : {list(best_item.assets.keys())[:10]}...")

In [ ]:
# =============================================================================
# [S3] 載入 6 波段 + SCL 雲遮罩 【COMPLETE — 直接執行】
# =============================================================================

# 選用 6 個反射率波段 + SCL（Scene Classification Layer）用於雲遮罩
BANDS_ALL = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]  # 分類用的 6 波段
BAND_NAMES = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

# 簽章（取得存取憑證）
signed_item = planetary_computer.sign(best_item)

# 使用 stackstac 堆疊（解析度統一為 20m）
# epsg=32651 = UTM 51N（台灣東岸），resolution 單位為公尺
stack = stackstac.stack(
    [signed_item],
    assets=BANDS_ALL,
    epsg=32651,
    resolution=20,
    bounds_latlon=HUALIEN_BBOX,
    dtype="float64",
)

print(f"Lazy 陣列形狀: {stack.shape}  (time, band, y, x)")
print("正在載入資料（.compute()），請稍候...")

# 實際載入資料
data = stack.compute()
img_all = data.values[0]  # shape: (7, H, W)

# --- 分離 SCL 與反射率波段 ---
scl = img_all[-1]           # 最後一個波段 = SCL
img = img_all[:-1] / 10000.0  # 前 6 波段轉為反射率

# --- SCL 雲遮罩 ---
# SCL 值定義（Sentinel-2 L2A）：
#   3 = Cloud Shadow, 8 = Cloud (medium prob), 9 = Cloud (high prob), 10 = Cirrus
cloud_mask = np.isin(scl, [3, 8, 9, 10])
n_cloud = int(np.sum(cloud_mask))
n_total = cloud_mask.size
print(f"\n=== SCL 雲遮罩 ===")
print(f"  雲/雲影像素: {n_cloud:,} / {n_total:,} ({n_cloud/n_total*100:.1f}%)")

# 套用雲遮罩（雲像素設為 NaN）
img[:, cloud_mask] = np.nan

# 將其他無效值（<= 0 或 > 1）也設為 NaN
img = np.where((img <= 0) | (img > 1), np.nan, img)

n_bands, n_rows, n_cols = img.shape

print(f"\n影像形狀: {img.shape}  (bands, height, width)")
print(f"像素大小: 20m x 20m")
print(f"影像尺寸: {n_cols * 20 / 1000:.1f} km x {n_rows * 20 / 1000:.1f} km")
print(f"有效像素比例（去雲後）: {np.sum(~np.isnan(img[0])) / img[0].size * 100:.1f}%")

for i, name in enumerate(BAND_NAMES):
    valid = img[i][~np.isnan(img[i])]
    if len(valid) > 0:
        print(
            f"  {name:6s} ({BANDS[i]}): "
            f"min={valid.min():.4f}, max={valid.max():.4f}, mean={valid.mean():.4f}"
        )

# 保存 UTM 座標資訊（後續 ROI 多邊形轉像素用）
x_coords = data.x.values
y_coords = data.y.values
x_res = float(x_coords[1] - x_coords[0])
y_res = float(y_coords[1] - y_coords[0])
img_transform = Affine(x_res, 0, float(x_coords[0]) - x_res/2,
                       0, y_res, float(y_coords[0]) - y_res/2)
print(f"\nUTM Transform 已建立（供 ROI 多邊形柵格化使用）")

In [ ]:
# =============================================================================
# [S4] RGB 真色彩 + 假色彩視覺化 【COMPLETE — 直接執行】
# =============================================================================


def percentile_stretch(band, low=2, high=98):
    """百分位拉伸：增強影像對比度"""
    valid = band[~np.isnan(band)]
    if len(valid) == 0:
        return band
    vmin = np.percentile(valid, low)
    vmax = np.percentile(valid, high)
    stretched = (band - vmin) / (vmax - vmin + 1e-10)
    return np.clip(stretched, 0, 1)


def make_rgb(img_array, band_indices, low=2, high=98):
    """組合 RGB 影像並做百分位拉伸"""
    rgb = np.stack(
        [percentile_stretch(img_array[i], low, high) for i in band_indices],
        axis=-1,
    )
    return np.nan_to_num(rgb, nan=0)


# 波段索引: B02=0(Blue), B03=1(Green), B04=2(Red),
#           B08=3(NIR),  B11=4(SWIR1), B12=5(SWIR2)
true_color = make_rgb(img, [2, 1, 0])    # R=B04, G=B03, B=B02
false_color = make_rgb(img, [3, 2, 1])   # R=B08, G=B04, B=B03

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(true_color)
axes[0].set_title("True Color (B4-B3-B2)\n真色彩影像", fontsize=14)
axes[0].set_xlabel("Column (pixel)")
axes[0].set_ylabel("Row (pixel)")

axes[1].imshow(false_color)
axes[1].set_title(
    "False Color (B8-B4-B3)\n假色彩影像（紅色＝植被）", fontsize=14
)
axes[1].set_xlabel("Column (pixel)")
axes[1].set_ylabel("Row (pixel)")

plt.suptitle(
    f"Sentinel-2 — {best_item.datetime.strftime('%Y-%m-%d')}  花蓮地區",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
plt.show()

## Lab 1: K-means 非監督分類

### 非監督式分類：讓演算法自動發現影像中的光譜群組

K-means 不需要訓練資料，直接根據像素的光譜特徵進行分群。
我們使用 6 個波段作為特徵空間，觀察不同 K 值的分類結果。

**步驟：**
1. 視覺化特徵空間（NIR vs Red 散佈圖）
2. 執行 K-means 分類
3. 分析各群組的光譜特徵
4. 比較不同 K 值的結果

In [ ]:
# =============================================================================
# [S6] 特徵空間視覺化 — NIR vs Red 散佈圖 【YOUR TURN】
# =============================================================================

# 波段索引提示:
#   B02=0(Blue), B03=1(Green), B04=2(Red),
#   B08=3(NIR),  B11=4(SWIR1), B12=5(SWIR2)

# ── YOUR TURN ──
# 從 img 陣列中取出 NIR 和 Red 波段，展平為一維陣列
# Hint: NIR 是第 3 個波段 (index=3)，Red 是第 2 個波段 (index=2)
nir_flat = img[___].flatten()   # 替換 ___ 為正確的波段索引
red_flat = img[___].flatten()   # 替換 ___ 為正確的波段索引
# ── END YOUR TURN ──

# 移除 NaN（已預先填好）
valid_mask_2d = ~(np.isnan(nir_flat) | np.isnan(red_flat))
nir_valid = nir_flat[valid_mask_2d]
red_valid = red_flat[valid_mask_2d]

# 隨機抽樣 10000 個像素（避免繪圖過慢）
np.random.seed(42)
n_sample = min(10000, len(nir_valid))
sample_idx = np.random.choice(len(nir_valid), size=n_sample, replace=False)

# 繪製散佈圖
fig, ax = plt.subplots(figsize=(8, 8))

# ── YOUR TURN ──
# 使用 ax.scatter() 繪製散佈圖
# Hint: x 軸 = red_valid[sample_idx], y 軸 = nir_valid[sample_idx]
# Hint: 設定 c="steelblue", alpha=0.15, s=3, edgecolors="none"
ax.scatter(
    ___,   # x 軸：Red 反射率（抽樣後）
    ___,   # y 軸：NIR 反射率（抽樣後）
    c="steelblue",
    alpha=0.15,
    s=3,
    edgecolors="none",
)
# ── END YOUR TURN ──

ax.set_xlabel("Red (B04) 反射率", fontsize=13)
ax.set_ylabel("NIR (B08) 反射率", fontsize=13)
ax.set_title("Feature Space: NIR vs Red\n特徵空間散佈圖", fontsize=14)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 0.6)

# 標註典型區域
ax.annotate(
    "植被 (高 NIR, 低 Red)",
    xy=(0.05, 0.4),
    fontsize=11,
    color="green",
    fontweight="bold",
)
ax.annotate(
    "裸地/崩塌 (高 Red, 低 NIR)",
    xy=(0.2, 0.15),
    fontsize=11,
    color="brown",
    fontweight="bold",
)
ax.annotate(
    "水體 (低 Red, 低 NIR)",
    xy=(0.01, 0.02),
    fontsize=11,
    color="blue",
    fontweight="bold",
)

# 對角線（soil line 參考）
ax.plot([0, 0.5], [0, 0.5], "k--", alpha=0.3, label="1:1 line")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("觀察：植被像素集中在左上方（高 NIR、低 Red），裸地/建物則靠近對角線。")
print("這種分群結構就是 K-means 要找的 pattern。")

In [ ]:
# =============================================================================
# [S7] K-means 分類 (K=5) 【YOUR TURN】
# =============================================================================

# --- 建立特徵矩陣（已預先填好）---
# 重塑為 (n_pixels, n_bands)
pixels = img.reshape(n_bands, -1).T  # shape: (n_pixels, 6)

# 找出有效像素（所有波段皆非 NaN）
valid_pixel_mask = ~np.any(np.isnan(pixels), axis=1)
pixels_valid = pixels[valid_pixel_mask]

print(f"總像素數    : {pixels.shape[0]:,}")
print(f"有效像素數  : {pixels_valid.shape[0]:,}")
print(f"特徵維度    : {pixels_valid.shape[1]} 波段")

# ── YOUR TURN ──
# 1. 建立 KMeans 模型並進行分類
# Hint: KMeans(n_clusters=5, random_state=42, n_init=10, max_iter=300)
K = 5
print(f"\n正在執行 K-means (K={K})...")

kmeans = ___  # 建立 KMeans 物件
labels_km = ___  # 使用 fit_predict() 對 pixels_valid 進行分類

print(f"K-means 完成！ Inertia = {kmeans.inertia_:.2f}")
# ── END YOUR TURN ──

# --- 重建分類圖（已預先填好）---
class_map_km = np.full(pixels.shape[0], np.nan)
class_map_km[valid_pixel_mask] = labels_km
class_map_km = class_map_km.reshape(n_rows, n_cols)

# ── YOUR TURN ──
# 2. 視覺化分類結果
# Hint: 使用 ax.imshow() 顯示 class_map_km
# Hint: 使用 plt.cm.get_cmap("Set1", K) 作為色彩映射
fig, ax = plt.subplots(figsize=(10, 8))
cmap_km = plt.cm.get_cmap("Set1", K)

im = ax.imshow(___, cmap=___, vmin=-0.5, vmax=K - 0.5)  # 填入正確的資料和色彩映射

cbar = plt.colorbar(im, ax=ax, ticks=range(K), shrink=0.8)
cbar.set_label("Cluster ID", fontsize=12)
ax.set_title(
    f"K-means 非監督式分類 (K={K})\nUnsupervised Classification", fontsize=14
)
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()
# ── END YOUR TURN ──

# 統計各群組像素數
print("\n各群組像素統計:")
for c in range(K):
    count = np.sum(labels_km == c)
    print(f"  Cluster {c}: {count:>8,} pixels ({count / len(labels_km) * 100:.1f}%)")

In [ ]:
# =============================================================================
# [S8] 各群組光譜特徵分析 【YOUR TURN】
# =============================================================================

# 繪圖設定（已預先填好）
fig, ax = plt.subplots(figsize=(10, 6))
colors_spec = plt.cm.Set1(np.linspace(0, 1, K))

# ── YOUR TURN ──
# 計算每個 cluster 的平均反射率，並繪製光譜曲線
# Hint: 對每個 cluster c (0 到 K-1)：
#   1. 用 pixels_valid[labels_km == c] 取出該 cluster 的像素
#   2. 用 .mean(axis=0) 計算平均光譜
#   3. 用 ax.plot() 繪製折線圖

cluster_means = np.zeros((K, n_bands))

for c in range(K):
    # 計算第 c 個 cluster 的平均反射率
    cluster_means[c] = ___  # Hint: pixels_valid[labels_km == c].mean(axis=0)

    # 繪製該 cluster 的光譜曲線
    ax.plot(
        range(n_bands),
        ___,  # Hint: cluster_means[c]
        "o-",
        color=colors_spec[c],
        linewidth=2,
        markersize=8,
        label=f"Cluster {c}",
    )
# ── END YOUR TURN ──

ax.set_xticks(range(n_bands))
ax.set_xticklabels(
    [f"{BAND_NAMES[i]}\n({BANDS[i]})" for i in range(n_bands)], fontsize=11
)
ax.set_ylabel("Mean Reflectance（平均反射率）", fontsize=13)
ax.set_xlabel("Band（波段）", fontsize=13)
ax.set_title("Cluster Mean Spectra\n各群組平均光譜特徵", fontsize=14)
ax.legend(fontsize=11, loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 答案驗證：預期光譜模式
print("\n=== 光譜特徵判讀指引（檢查你的結果是否合理）===")
print("  - NIR 高、Red 低 → 健康植被")
print("  - 所有波段都低   → 水體")
print("  - Red/SWIR 高    → 裸地 / 崩塌地")
print("  - 整體偏亮       → 建物 / 人造地物")
print("\n提示：如果你的圖中有一條曲線在 NIR 處有明顯高峰，那很可能是植被 cluster。")

### 課堂練習：你的 cluster 是什麼地物？

根據上方的光譜分析結果，請嘗試將每個 cluster 對應到可能的土地覆蓋類型。

| Cluster | 你觀察到的光譜特徵 | 推測的地物類型 |
|---------|-------------------|---------------|
| 0 | （在此填寫） | （在此填寫） |
| 1 | （在此填寫） | （在此填寫） |
| 2 | （在此填寫） | （在此填寫） |
| 3 | （在此填寫） | （在此填寫） |
| 4 | （在此填寫） | （在此填寫） |

**思考題：**

1. 非監督分類的 cluster 編號沒有固定意義，每次執行結果可能不同。你能確定每個 cluster 一定只對應一種地物嗎？為什麼？

2. 觀察分類圖：陸地上是否有和海洋相同顏色的像素？如果有，你認為那些像素實際上是什麼？（提示：想想雲影和地形陰影的光譜特徵）

3. K-means 為什麼會把陰影和水體分到同一類？這暗示了非監督分類的什麼限制？

In [ ]:
# =============================================================================
# [S10] K 值比較 【YOUR TURN】
# =============================================================================
# 嘗試不同的 K 值，觀察分類結果的差異

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, k_val in enumerate([5, 7, 10]):
    # ── YOUR TURN ──
    # 對每個 k_val 執行 KMeans 分類
    # Hint: KMeans(n_clusters=k_val, random_state=42, n_init=10, max_iter=300)
    km_temp = ___  # 建立 KMeans 物件
    lbl = ___      # 使用 fit_predict() 對 pixels_valid 分類
    # ── END YOUR TURN ──

    # 重建分類圖並視覺化（已預先填好）
    cmap_temp = plt.cm.get_cmap("Set1", k_val)
    cm_temp = np.full(pixels.shape[0], np.nan)
    cm_temp[valid_pixel_mask] = lbl
    cm_temp = cm_temp.reshape(n_rows, n_cols)
    axes[ax_idx].imshow(cm_temp, cmap=cmap_temp, vmin=-0.5, vmax=k_val - 0.5)
    axes[ax_idx].set_title(f"K = {k_val}", fontsize=13)
    axes[ax_idx].set_xlabel("Column")

plt.suptitle("K-means：不同 K 值比較", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("思考：K 值越大，分類越細緻，但過多的群組可能難以解讀。")
print("你認為哪個 K 值最適合這個研究區域？")
print()
print("=" * 65)
print("  觀察重點：注意陸地上是否有和海洋同色的像素")
print("=" * 65)
print("  這些陸地上的「疑似水體」像素可能是：")
print("    1. 雲影（cloud shadow）— SCL 遮罩未完全移除的殘留")
print("    2. 山區地形陰影 — 中央山脈東坡的自遮蔽效應")
print("    3. 深色建物屋頂 — 20m 解析度下的混合像素")
print()
print("  ⚠️  這正是 K-means 的根本限制：")
print("    非監督分類只看光譜相似度，無法區分「暗因為是水」")
print("    和「暗因為是陰影」。後續 Lab 2 的監督式分類可以")
print("    利用訓練樣本學到水體和陰影在 SWIR 波段的細微差異。")

## Lab 2: Random Forest 監督分類

### 監督式分類：使用訓練樣本建立 Random Forest 分類模型

需要事先定義各類別的訓練區域（ROI），再讓模型學習光譜特徵與類別的對應關係。

**你的任務：** 在 Google Earth 上繪製 5 類地物的多邊形 ROI，匯出為 KMZ，上傳到 Colab。

| Class | 名稱 | 說明 | 選取建議 |
|-------|------|------|---------|
| 0 | 水體 | 海洋、河流 | 花蓮外海明顯的深藍色區域 |
| 1 | 森林 | 山坡密林 | 中央山脈東側，深綠色區域 |
| 2 | 農地 | 水稻田等 | 花蓮縱谷，淺綠色規則區塊 |
| 3 | 裸地/崩塌 | 河床、崩塌面 | 花蓮溪河床、山區裸露面 |
| 4 | 建物 | 市區建築 | 花蓮市中心，灰白色區域 |

**KMZ 製作步驟：**
1. 開啟 Google Earth Pro
2. 對每一類地物，使用「新增多邊形」工具框選代表性區域
3. **多邊形命名必須是：** `水體`、`森林`、`農地`、`裸露地`、`建物`（中文名稱）
4. 每類建議畫 2–3 個多邊形，越大越好（至少 50 個像素 ≈ 200m × 200m）
5. 全選 → 右鍵 → 「將地點另存為...」→ 存成 `.kmz`

**步驟：**
1. 上傳 KMZ → 自動解析 ROI
2. 訓練 Random Forest 模型
3. 對全影像進行分類預測
4. 評估分類精度（混淆矩陣、OA、Kappa）
5. 後處理（中值濾波）

In [ ]:
# =============================================================================
# [S12] 上傳 KMZ + 解析 ROI 訓練樣本 【YOUR TURN — 上傳你的 KMZ】
# =============================================================================
# 步驟：
#   1. 在 Google Earth Pro 繪製多邊形 ROI（見上方說明）
#   2. 匯出為 KMZ 檔案
#   3. 執行此 cell，會跳出上傳對話框
#   4. 程式自動解析多邊形 → 轉 UTM → 柵格化 → 提取訓練像素

from google.colab import files

# --- 類別定義 ---
CLASS_NAMES = ["Water", "Forest", "Agriculture", "Bare/Landslide", "Built-up"]
CLASS_NAMES_ZH = ["水體", "森林", "農地", "裸地/崩塌", "建物"]
N_CLASSES = len(CLASS_NAMES)

# KMZ 中的多邊形名稱 → 類別 ID 對照表
# （你在 Google Earth 裡的多邊形名稱必須包含以下關鍵字之一）
name_to_class = {
    "水體": 0, "水": 0, "Water": 0, "water": 0,
    "森林": 1, "林": 1, "Forest": 1, "forest": 1,
    "農地": 2, "農": 2, "Agriculture": 2, "agriculture": 2, "Cropland": 2,
    "裸露地": 3, "裸地": 3, "崩塌": 3, "裸": 3, "Bare": 3, "bare": 3, "Landslide": 3,
    "建物": 4, "建": 4, "Built": 4, "built": 4, "Built-up": 4,
}

# ── YOUR TURN ──
# 上傳你的 KMZ 檔案（執行此 cell 會出現上傳按鈕）
print("請上傳你在 Google Earth 製作的 KMZ 檔案...")
uploaded = files.upload()
kmz_filename = list(uploaded.keys())[0]
print(f"已上傳: {kmz_filename}")
# ── END YOUR TURN ──

# --- 解析 KMZ（自動處理，不需修改）---
def parse_kmz(filename):
    """解析 KMZ 檔案，回傳 {class_id: [Polygon, ...]} 字典"""
    ns = {"kml": "http://www.opengis.net/kml/2.2"}
    polygons_by_class = {i: [] for i in range(N_CLASSES)}

    with zipfile.ZipFile(filename, 'r') as z:
        kml_name = [n for n in z.namelist() if n.endswith('.kml')][0]
        with z.open(kml_name) as f:
            tree = ET.parse(f)

    for pm in tree.findall(".//{http://www.opengis.net/kml/2.2}Placemark"):
        name_elem = pm.find("{http://www.opengis.net/kml/2.2}name")
        coords_elem = pm.find(".//{http://www.opengis.net/kml/2.2}coordinates")
        if name_elem is None or coords_elem is None:
            continue

        # 比對名稱 → 類別 ID
        pm_name = name_elem.text.strip()
        cls_id = None
        for keyword, cid in name_to_class.items():
            if keyword in pm_name:
                cls_id = cid
                break
        if cls_id is None:
            print(f"  ⚠️ 無法辨識類別: '{pm_name}'，已跳過")
            continue

        # 解析座標
        coords_text = coords_elem.text.strip()
        points = []
        for c in coords_text.split():
            parts = c.split(",")
            points.append((float(parts[0]), float(parts[1])))

        if len(points) >= 3:
            poly = Polygon(points)
            if poly.is_valid and poly.area > 0:
                polygons_by_class[cls_id].append(poly)
                print(f"  ✓ {pm_name} → {CLASS_NAMES[cls_id]}")

    return polygons_by_class

print("\n--- 解析 KMZ ---")
polygons_by_class = parse_kmz(kmz_filename)

# --- 轉 UTM + 柵格化（自動處理）---
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)
shapes_for_rasterize = []

for cls_id, poly_list in polygons_by_class.items():
    for poly in poly_list:
        coords_utm = [transformer.transform(x, y) for x, y in poly.exterior.coords]
        poly_utm = Polygon(coords_utm)
        if poly_utm.is_valid and poly_utm.area > 0:
            shapes_for_rasterize.append((poly_utm, cls_id))

roi_raster = rasterize(
    shapes_for_rasterize,
    out_shape=(n_rows, n_cols),
    transform=img_transform,
    fill=-1,
    dtype=np.int8,
    all_touched=True,
)

# --- 提取訓練像素 ---
X_train_list = []
y_train_list = []

for cls_id in range(N_CLASSES):
    cls_mask = (roi_raster == cls_id)
    cls_pixels = img[:, cls_mask].T
    valid_rows = ~np.any(np.isnan(cls_pixels), axis=1)
    cls_valid = cls_pixels[valid_rows]
    if len(cls_valid) > 0:
        X_train_list.append(cls_valid)
        y_train_list.append(np.full(len(cls_valid), cls_id))

X_roi = np.vstack(X_train_list)
y_roi = np.concatenate(y_train_list)

# --- 統計報告 ---
print("\n=== 訓練樣本統計 ===")
print(f"總訓練像素數: {len(y_roi):,}\n")
for cls_id in range(N_CLASSES):
    count = int(np.sum(y_roi == cls_id))
    n_poly = len(polygons_by_class[cls_id])
    status = "✅" if count >= 30 else "⚠️ 偏少，建議增加 ROI"
    print(
        f"  Class {cls_id} ({CLASS_NAMES[cls_id]:>16s} / "
        f"{CLASS_NAMES_ZH[cls_id]}): {count:>6,} pixels, "
        f"{n_poly} 個多邊形  {status}"
    )

# --- 視覺化 ROI ---
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(make_rgb(img, [2, 1, 0]))

roi_colors_vis = ["#1f78b4", "#33a02c", "#b2df8a", "#d2691e", "#e31a1c"]
for cls_id in range(N_CLASSES):
    mask = (roi_raster == cls_id)
    if np.any(mask):
        overlay = np.zeros((*mask.shape, 4))
        c = mcolors.to_rgba(roi_colors_vis[cls_id], alpha=0.5)
        overlay[mask] = c
        ax.imshow(overlay)

legend_roi = [Patch(facecolor=roi_colors_vis[i], alpha=0.6,
                    label=f"{CLASS_NAMES[i]} ({CLASS_NAMES_ZH[i]})")
              for i in range(N_CLASSES)]
ax.legend(handles=legend_roi, loc="upper right", fontsize=10, framealpha=0.9)
ax.set_title("Your Training ROI\n你的訓練樣本位置", fontsize=14)
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

print("\n檢查你的 ROI 是否合理：")
print("  - 水體 ROI 是否確實在海面或河流上？")
print("  - 森林 ROI 是否在山坡密林區？")
print("  - 每類是否至少有 30+ 像素？")

### 🔍 ROI 智慧驗證（Smart ROI Validation）

下方的驗證程式會自動檢查你的訓練樣本品質，模擬專家審查流程：

| 檢查項目 | 說明 |
|---------|------|
| **樣本數量** | 每類至少需要 30 個像素（建議 100+），依特徵數給出建議值 |
| **光譜合理性** | 檢查各類別的光譜特徵是否符合預期（如水體 NIR 應低、森林 NDVI 應高） |
| **類別可分離性** | 各類別之間光譜是否足夠不同（Jeffries-Matusita 距離） |
| **空間分布** | ROI 是否過度集中在影像某區域 |

執行後會產生一份「診斷報告」，並針對不合格的項目給出具體改善建議。

In [ ]:
# =============================================================================
# [S12b] ROI 智慧驗證 — 自動診斷訓練樣本品質 【COMPLETE — 直接執行】
# =============================================================================
# 此程式模擬遙測專家的審查流程，自動檢查你的 ROI 是否適合訓練分類模型。
# 不需修改任何程式碼，直接執行即可看到診斷報告。

import textwrap

# ===================== 1. 樣本數量檢查 =====================
print("=" * 70)
print("📊  檢查 1／4：樣本數量")
print("=" * 70)

# Random Forest 經驗法則：每類至少 10 × 特徵數（6 波段 → 60），建議 100+
MIN_PIXELS = 30       # 最低門檻
REC_PIXELS = 100      # 建議值
GOOD_PIXELS = 500     # 理想值

qty_issues = []
for cls_id in range(N_CLASSES):
    count = int(np.sum(y_roi == cls_id))
    if count == 0:
        grade = "❌ 無樣本"
        qty_issues.append((cls_id, "missing"))
    elif count < MIN_PIXELS:
        grade = f"⚠️ 不足（最少 {MIN_PIXELS}）"
        qty_issues.append((cls_id, "low"))
    elif count < REC_PIXELS:
        grade = f"🔶 偏少（建議 {REC_PIXELS}+）"
        qty_issues.append((cls_id, "marginal"))
    elif count < GOOD_PIXELS:
        grade = "✅ 合格"
    else:
        grade = "✅ 充足"
    print(f"  {CLASS_NAMES[cls_id]:>16s} ({CLASS_NAMES_ZH[cls_id]}): "
          f"{count:>6,} pixels  {grade}")

# ===================== 2. 光譜合理性檢查 =====================
print(f"\n{'=' * 70}")
print("🌈  檢查 2／4：光譜合理性（各類別光譜是否符合物理預期）")
print("=" * 70)

# 波段索引: B02=0, B03=1, B04=2(Red), B08=3(NIR), B11=4(SWIR1), B12=5(SWIR2)
spectral_issues = []

# 計算各類別的波段平均值
class_means = {}
for cls_id in range(N_CLASSES):
    mask_cls = (y_roi == cls_id)
    if np.sum(mask_cls) > 0:
        class_means[cls_id] = X_roi[mask_cls].mean(axis=0)
    else:
        class_means[cls_id] = None

# 定義光譜規則（基於遙測物理原理）
spectral_rules = {
    0: [  # Water
        ("NIR 應低（< 0.10）", lambda m: m[3] < 0.10,
         "水體強烈吸收近紅外光，NIR 反射率應很低。你的水體 ROI 可能包含了陸地像素。"),
        ("SWIR 應低（< 0.08）", lambda m: m[4] < 0.08,
         "水體在 SWIR 波段幾乎完全吸收。如果偏高，可能選到了淺水區或河岸。"),
        ("Blue > NIR", lambda m: m[0] > m[3],
         "水體的 Blue 反射率通常高於 NIR。如果不是，可能 ROI 不在水面上。"),
    ],
    1: [  # Forest
        ("NDVI 應高（> 0.4）", lambda m: (m[3] - m[2]) / (m[3] + m[2] + 1e-10) > 0.4,
         "健康森林的 NDVI 通常在 0.5-0.8。偏低可能選到了稀疏植被或混合像素。"),
        ("NIR 應高（> 0.15）", lambda m: m[3] > 0.15,
         "森林的 NIR 反射率應較高（植被結構散射）。"),
        ("Red 應低（< 0.10）", lambda m: m[2] < 0.10,
         "森林強烈吸收紅光用於光合作用。Red 偏高可能選到了枯萎或裸露區。"),
    ],
    2: [  # Agriculture
        ("NDVI 應為正（> 0.2）", lambda m: (m[3] - m[2]) / (m[3] + m[2] + 1e-10) > 0.2,
         "農地（即使休耕）通常有些植被覆蓋。NDVI 太低可能選到了裸露田地。"),
        ("NIR 應適中", lambda m: m[3] > 0.10,
         "農地的 NIR 反射率應至少適中。"),
    ],
    3: [  # Bare/Landslide
        ("NDVI 應低（< 0.3）", lambda m: (m[3] - m[2]) / (m[3] + m[2] + 1e-10) < 0.3,
         "裸露地表幾乎沒有植被，NDVI 應接近 0 或為負。偏高可能選到了植被恢復區。"),
        ("Red 或 SWIR 應可見", lambda m: m[2] > 0.05 or m[4] > 0.05,
         "裸地在可見光和 SWIR 都有一定反射率。"),
    ],
    4: [  # Built-up
        ("NDBI 應為正或接近 0", lambda m: (m[4] - m[3]) / (m[4] + m[3] + 1e-10) > -0.15,
         "建物的 NDBI（SWIR-NIR 差異）通常高於植被。如果太負，可能選到了公園綠地。"),
        ("整體反射率應適中",
         lambda m: np.mean([m[0], m[1], m[2]]) > 0.04,
         "建物（混凝土、鐵皮屋頂）在可見光有一定反射率。太低可能是陰影或水域。"),
    ],
}

for cls_id in range(N_CLASSES):
    m = class_means[cls_id]
    if m is None:
        print(f"\n  {CLASS_NAMES[cls_id]}: ❌ 無樣本，無法檢查")
        continue

    print(f"\n  {CLASS_NAMES[cls_id]} ({CLASS_NAMES_ZH[cls_id]}):")
    ndvi = (m[3] - m[2]) / (m[3] + m[2] + 1e-10)
    print(f"    平均光譜: Blue={m[0]:.4f} Green={m[1]:.4f} Red={m[2]:.4f} "
          f"NIR={m[3]:.4f} SWIR1={m[4]:.4f} SWIR2={m[5]:.4f}")
    print(f"    NDVI={ndvi:.3f}")

    rules = spectral_rules.get(cls_id, [])
    all_pass = True
    for rule_name, check_fn, advice in rules:
        passed = check_fn(m)
        if passed:
            print(f"    ✅ {rule_name}")
        else:
            print(f"    ⚠️ {rule_name} — 未通過")
            print(f"       💡 {advice}")
            spectral_issues.append((cls_id, rule_name))
            all_pass = False
    if all_pass and len(rules) > 0:
        print(f"    → 光譜特徵合理 👍")

# ===================== 3. 類別可分離性分析 =====================
print(f"\n{'=' * 70}")
print("📐  檢查 3／4：類別可分離性（各類之間是否容易區分）")
print("=" * 70)

# 使用簡化版 Jeffries-Matusita 距離
# JM ∈ [0, 2]：> 1.9 優秀，1.5-1.9 良好，< 1.5 需注意
separability_issues = []

def bhattacharyya_distance(X1, X2):
    """計算兩組樣本的 Bhattacharyya 距離（簡化版）"""
    if len(X1) < 2 or len(X2) < 2:
        return 0.0
    m1, m2 = X1.mean(axis=0), X2.mean(axis=0)
    s1, s2 = np.cov(X1.T) + np.eye(X1.shape[1]) * 1e-6, \
             np.cov(X2.T) + np.eye(X2.shape[1]) * 1e-6
    s_avg = (s1 + s2) / 2
    try:
        sign, logdet = np.linalg.slogdet(s_avg)
        sign1, logdet1 = np.linalg.slogdet(s1)
        sign2, logdet2 = np.linalg.slogdet(s2)
        if sign <= 0 or sign1 <= 0 or sign2 <= 0:
            return 0.0
        dm = m1 - m2
        term1 = 0.125 * dm @ np.linalg.solve(s_avg, dm)
        term2 = 0.5 * (logdet - 0.5 * (logdet1 + logdet2))
        bd = term1 + term2
        jm = 2 * (1 - np.exp(-bd))
        return min(jm, 2.0)
    except np.linalg.LinAlgError:
        return 0.0

print(f"\n  {'':20s}", end="")
for j in range(N_CLASSES):
    print(f"{CLASS_NAMES[j][:8]:>10s}", end="")
print()

for i in range(N_CLASSES):
    print(f"  {CLASS_NAMES[i]:20s}", end="")
    for j in range(N_CLASSES):
        if i == j:
            print(f"{'---':>10s}", end="")
        elif j > i:
            Xi = X_roi[y_roi == i]
            Xj = X_roi[y_roi == j]
            if len(Xi) < 2 or len(Xj) < 2:
                print(f"{'N/A':>10s}", end="")
            else:
                jm = bhattacharyya_distance(Xi, Xj)
                if jm >= 1.9:
                    label = f"{jm:.2f} ✅"
                elif jm >= 1.5:
                    label = f"{jm:.2f} 🔶"
                else:
                    label = f"{jm:.2f} ⚠️"
                    separability_issues.append(
                        (CLASS_NAMES[i], CLASS_NAMES[j], jm))
                print(f"{label:>10s}", end="")
        else:
            print(f"{'':>10s}", end="")
    print()

print(f"\n  判讀：JM 距離 > 1.9 = 容易區分，1.5-1.9 = 尚可，< 1.5 = 易混淆")
if separability_issues:
    print(f"\n  ⚠️ 以下類別對容易混淆：")
    for c1, c2, jm in separability_issues:
        print(f"    • {c1} vs {c2} (JM={jm:.2f})")
        print(f"      💡 建議：選擇更典型的 ROI 區域，或考慮合併這兩類")
else:
    print(f"\n  ✅ 所有類別對的可分離性良好")

# ===================== 4. 空間分布檢查 =====================
print(f"\n{'=' * 70}")
print("🗺️  檢查 4／4：ROI 空間分布（是否過度集中）")
print("=" * 70)

spatial_issues = []
img_center_r, img_center_c = n_rows // 2, n_cols // 2
img_diag = np.sqrt(n_rows**2 + n_cols**2)

for cls_id in range(N_CLASSES):
    mask_cls = (roi_raster == cls_id)
    if not np.any(mask_cls):
        continue
    rows_cls, cols_cls = np.where(mask_cls)
    # 計算 ROI 的空間範圍相對於影像大小
    row_span = (rows_cls.max() - rows_cls.min()) / n_rows
    col_span = (cols_cls.max() - cols_cls.min()) / n_cols
    # 如果所有 ROI 像素都集中在影像的一小塊區域
    spread = max(row_span, col_span)
    n_poly = len(polygons_by_class[cls_id])

    if n_poly == 1 and spread < 0.05:
        status = "⚠️ 僅 1 個小區域，空間代表性不足"
        spatial_issues.append((cls_id, "single_small"))
    elif spread < 0.02:
        status = "⚠️ ROI 過度集中"
        spatial_issues.append((cls_id, "concentrated"))
    elif n_poly >= 2:
        status = f"✅ {n_poly} 個多邊形，分布合理"
    else:
        status = "🔶 建議增加更多分散的 ROI"

    print(f"  {CLASS_NAMES[cls_id]:>16s}: 空間範圍 {spread*100:.1f}% 影像寬度, "
          f"{n_poly} 個多邊形  {status}")

# ===================== 綜合評估報告 =====================
print(f"\n{'=' * 70}")
print("📋  綜合評估報告")
print("=" * 70)

total_issues = len(qty_issues) + len(spectral_issues) + \
               len(separability_issues) + len(spatial_issues)

if total_issues == 0:
    print("\n  🎉 你的 ROI 品質優秀！所有檢查項目都通過了。")
    print("     可以放心進入下一步訓練模型。")
elif total_issues <= 3:
    print(f"\n  👍 大致良好，發現 {total_issues} 個小問題：")
else:
    print(f"\n  ⚠️ 發現 {total_issues} 個問題，建議改善後再訓練模型：")

# 具體改善建議
suggestions = []
for cls_id, level in qty_issues:
    if level == "missing":
        suggestions.append(
            f"• {CLASS_NAMES[cls_id]}：完全沒有樣本！請在 Google Earth 中為此類別"
            f"畫至少 2 個多邊形。")
    elif level == "low":
        suggestions.append(
            f"• {CLASS_NAMES[cls_id]}：像素數不足 {MIN_PIXELS}。"
            f"請畫更大的多邊形或增加更多 ROI 區域。")
    elif level == "marginal":
        suggestions.append(
            f"• {CLASS_NAMES[cls_id]}：像素數偏少。"
            f"建議擴大 ROI 面積至 {REC_PIXELS}+ 像素以提高模型穩定性。")

for cls_id, rule_name in spectral_issues:
    suggestions.append(
        f"• {CLASS_NAMES[cls_id]}：光譜檢查「{rule_name}」未通過。"
        f"請確認 ROI 位置是否正確（可對照真色彩影像）。")

for c1, c2, jm in separability_issues:
    suggestions.append(
        f"• {c1} 與 {c2} 光譜相似（JM={jm:.2f}），"
        f"可能導致混淆。嘗試選擇更具代表性的典型區域。")

for cls_id, issue_type in spatial_issues:
    suggestions.append(
        f"• {CLASS_NAMES[cls_id]}：ROI 空間分布過於集中。"
        f"建議在不同位置多畫幾個多邊形，提高空間代表性。")

for s in suggestions:
    print(f"  {s}")

if total_issues > 0:
    print(f"\n  修改 KMZ 後重新執行 [S12] 和此 cell 即可更新診斷。")
else:
    print(f"\n  ✅ 繼續執行下方的 [S13] 開始訓練 Random Forest 模型吧！")

# ===================== 光譜剖面圖 =====================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左圖：各類別平均光譜
ax = axes[0]
cls_colors = ["#1f78b4", "#33a02c", "#b2df8a", "#d2691e", "#e31a1c"]
for cls_id in range(N_CLASSES):
    m = class_means[cls_id]
    if m is not None:
        ax.plot(range(n_bands), m, "o-", color=cls_colors[cls_id],
                linewidth=2, markersize=7,
                label=f"{CLASS_NAMES[cls_id]} ({CLASS_NAMES_ZH[cls_id]})")
ax.set_xticks(range(n_bands))
ax.set_xticklabels([f"{BAND_NAMES[i]}\n({BANDS[i]})" for i in range(n_bands)],
                   fontsize=9)
ax.set_ylabel("平均反射率", fontsize=12)
ax.set_title("各類別平均光譜曲線\nClass Mean Spectra", fontsize=13)
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)

# 右圖：各類別 NDVI 箱型圖
ax2 = axes[1]
ndvi_by_class = []
labels_bp = []
for cls_id in range(N_CLASSES):
    mask_cls = (y_roi == cls_id)
    if np.sum(mask_cls) > 0:
        cls_data = X_roi[mask_cls]
        ndvi_cls = (cls_data[:, 3] - cls_data[:, 2]) / \
                   (cls_data[:, 3] + cls_data[:, 2] + 1e-10)
        ndvi_by_class.append(ndvi_cls)
        labels_bp.append(f"{CLASS_NAMES_ZH[cls_id]}")
    else:
        ndvi_by_class.append([0])
        labels_bp.append(f"{CLASS_NAMES_ZH[cls_id]}*")

bp = ax2.boxplot(ndvi_by_class, labels=labels_bp, patch_artist=True,
                 widths=0.6, showfliers=True,
                 flierprops=dict(marker=".", markersize=3, alpha=0.3))
for patch, color in zip(bp["boxes"], cls_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel("NDVI", fontsize=12)
ax2.set_title("各類別 NDVI 分布\nNDVI Distribution by Class", fontsize=13)
ax2.axhline(y=0.3, color="gray", linestyle="--", alpha=0.5, label="NDVI=0.3")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n左圖：如果兩條曲線形狀幾乎重疊，表示這兩類容易混淆。")
print("右圖：NDVI 分布應該——水體最低、裸地次低、農地中等、森林最高。")

In [ ]:
# =============================================================================
# [S13] 訓練 Random Forest 模型 【YOUR TURN】
# =============================================================================

# 80/20 分割，保持類別比例（已預先填好）
X_tr, X_te, y_tr, y_te = train_test_split(
    X_roi, y_roi, test_size=0.2, random_state=42, stratify=y_roi
)

print(f"訓練集: {len(y_tr):,} 像素")
print(f"測試集: {len(y_te):,} 像素")

# ── YOUR TURN ──
# 1. 建立 Random Forest 分類器
# Hint: RandomForestClassifier(n_estimators=200, max_features="sqrt",
#                              random_state=42, n_jobs=-1, oob_score=True)

rf = ___  # 建立 RandomForestClassifier 物件

# 2. 訓練模型
# Hint: 使用 rf.fit(X_tr, y_tr)
print("\n正在訓練 Random Forest（n_estimators=200）...")
___  # 訓練模型

# 3. 預測測試集並計算精度
# Hint: rf.predict(X_te) 和 accuracy_score(y_te, y_pred_test)
y_pred_test = ___    # 對測試集進行預測
test_acc = ___       # 計算測試集精度
oob_acc = rf.oob_score_  # OOB 精度（Random Forest 內建）

print(f"\n=== 模型評估 ===")
print(f"  Test Accuracy : {test_acc:.4f} ({test_acc * 100:.2f}%)")
print(f"  OOB  Accuracy : {oob_acc:.4f} ({oob_acc * 100:.2f}%)")
# ── END YOUR TURN ──

print("\n=== Classification Report ===")
print(classification_report(y_te, y_pred_test, target_names=CLASS_NAMES, digits=4))

In [ ]:
# =============================================================================
# [S14] 全影像分類 + 分類圖視覺化 【YOUR TURN】
# =============================================================================

# --- 分類配色設定（已預先填好）---
lc_colors = [
    "#1f78b4",  # 0: Water — 藍色
    "#33a02c",  # 1: Forest — 深綠
    "#b2df8a",  # 2: Agriculture — 淺綠
    "#d2691e",  # 3: Bare/Landslide — 棕色
    "#e31a1c",  # 4: Built-up — 紅色
]
lc_cmap = mcolors.ListedColormap(lc_colors)
lc_bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
lc_norm = mcolors.BoundaryNorm(lc_bounds, lc_cmap.N)

# ── YOUR TURN ──
# 1. 對所有有效像素進行分類預測
# Hint: 使用 rf.predict(pixels_valid)
print("正在對全影像進行分類預測...")
rf_labels = ___  # 對 pixels_valid 進行預測

# 2. 重建分類圖（將一維結果還原為二維影像）
# Hint: 先建立全 NaN 陣列，再填入預測結果，最後 reshape
class_map_rf = np.full(pixels.shape[0], np.nan)
class_map_rf[valid_pixel_mask] = ___  # 填入預測結果
class_map_rf = class_map_rf.reshape(___, ___)  # 填入正確的行列數
# ── END YOUR TURN ──

# --- 視覺化（已預先填好）---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 左：真色彩
axes[0].imshow(true_color)
axes[0].set_title("True Color Reference\n真色彩參考影像", fontsize=13)

# 右：分類結果
im = axes[1].imshow(class_map_rf, cmap=lc_cmap, norm=lc_norm)
axes[1].set_title("Random Forest Classification\n隨機森林分類結果", fontsize=13)

# 圖例
legend_patches = [
    Patch(facecolor=lc_colors[i], label=f"{CLASS_NAMES[i]} ({CLASS_NAMES_ZH[i]})")
    for i in range(N_CLASSES)
]
axes[1].legend(
    handles=legend_patches,
    loc="lower right",
    fontsize=10,
    framealpha=0.9,
    fancybox=True,
)

for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.suptitle(
    "Supervised Classification Result\n監督式分類成果", fontsize=15, y=1.02
)
plt.tight_layout()
plt.show()
print("分類完成。")

In [ ]:
# =============================================================================
# [S15] 混淆矩陣 + OA / Kappa 【YOUR TURN】
# =============================================================================

# 計算混淆矩陣（已預先填好）
cm = confusion_matrix(y_te, y_pred_test)

# ── YOUR TURN ──
# 1. 繪製混淆矩陣熱力圖
# Hint: 使用 ax.imshow(cm, cmap="Blues") 顯示混淆矩陣
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(___, interpolation="nearest", cmap="Blues")  # 填入混淆矩陣
plt.colorbar(im, ax=ax, shrink=0.8)

# 在格子中標註數值
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(
            j, i, f"{cm[i, j]}",
            ha="center", va="center",
            fontsize=12, fontweight="bold", color=color,
        )

ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(CLASS_NAMES, fontsize=11)
ax.set_xlabel("Predicted（預測類別）", fontsize=13)
ax.set_ylabel("True（實際類別）", fontsize=13)
ax.set_title("Confusion Matrix\n混淆矩陣", fontsize=14)
plt.tight_layout()
plt.show()
# ── END YOUR TURN ──

# ── YOUR TURN ──
# 2. 計算 Overall Accuracy (OA) 和 Cohen's Kappa
# Hint: accuracy_score(y_te, y_pred_test) 和 cohen_kappa_score(y_te, y_pred_test)
OA = ___     # 計算 Overall Accuracy
kappa = ___  # 計算 Cohen's Kappa

print(f"\nOverall Accuracy (OA) : {OA:.4f} ({OA * 100:.2f}%)")
print(f"Cohen's Kappa         : {kappa:.4f}")
print(f"\n判讀：Kappa > 0.8 為 almost perfect agreement")
# ── END YOUR TURN ──

In [ ]:
# =============================================================================
# [S16] 特徵重要性分析 【COMPLETE — 直接執行】
# =============================================================================

importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ["#2166ac", "#67a9cf", "#d6604d", "#1b7837", "#f4a582", "#b2182b"]

bars = ax.bar(
    range(n_bands),
    importances[sorted_idx],
    color=[bar_colors[i] for i in sorted_idx],
    edgecolor="black",
    linewidth=0.5,
)

ax.set_xticks(range(n_bands))
ax.set_xticklabels(
    [f"{BAND_NAMES[i]}\n({BANDS[i]})" for i in sorted_idx], fontsize=11
)
ax.set_ylabel("Feature Importance（特徵重要性）", fontsize=13)
ax.set_title("Random Forest Feature Importance\n隨機森林各波段重要性", fontsize=14)
ax.grid(axis="y", alpha=0.3)

# 在柱狀圖上標數值
for bar_obj, imp in zip(bars, importances[sorted_idx]):
    ax.text(
        bar_obj.get_x() + bar_obj.get_width() / 2,
        bar_obj.get_height() + 0.005,
        f"{imp:.3f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

plt.tight_layout()
plt.show()

print("\n=== 特徵重要性排序 ===")
for rank, idx in enumerate(sorted_idx):
    print(f"  {rank + 1}. {BAND_NAMES[idx]:6s} ({BANDS[idx]}): {importances[idx]:.4f}")

print("\n分析：NIR 和 SWIR 波段通常對土地覆蓋分類最重要，")
print("因為它們能有效區分植被、水體和裸露地表。")

In [ ]:
# =============================================================================
# [S17] 後處理 — 中值濾波 【YOUR TURN】
# =============================================================================

# ── YOUR TURN ──
# 1. 將分類圖中的 NaN 暫時填充為 -1，轉為整數
# Hint: np.nan_to_num(class_map_rf, nan=-1).astype(np.int8)
map_for_filter = ___

# 2. 套用 3x3 中值濾波
# Hint: median_filter(map_for_filter, size=3)
map_filtered = ___

# 3. 恢復 NaN
class_map_filtered = map_filtered.astype(float)
class_map_filtered[map_for_filter == -1] = np.nan
# ── END YOUR TURN ──

# 計算變更像素數（已預先填好）
valid_comparison = (~np.isnan(class_map_rf)) & (~np.isnan(class_map_filtered))
pixels_changed = int(
    np.sum(class_map_rf[valid_comparison] != class_map_filtered[valid_comparison])
)
total_valid = int(np.sum(valid_comparison))

# --- 前後比較 ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(class_map_rf, cmap=lc_cmap, norm=lc_norm)
axes[0].set_title("Before Filtering\n濾波前（原始分類）", fontsize=13)

axes[1].imshow(class_map_filtered, cmap=lc_cmap, norm=lc_norm)
axes[1].set_title("After Median Filter (3x3)\n中值濾波後", fontsize=13)

for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

# 共用圖例
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=5,
    fontsize=10,
    framealpha=0.9,
    bbox_to_anchor=(0.5, -0.02),
)

plt.suptitle("Post-processing: Median Filter\n後處理：中值濾波平滑", fontsize=15)
plt.tight_layout()
plt.show()

print(f"變更像素數: {pixels_changed:,} / {total_valid:,} ({pixels_changed / total_valid * 100:.2f}%)")
print("中值濾波可消除零碎的 salt-and-pepper 雜訊，使分類結果更平滑。")

## 反思問題

完成以上實作後，請思考並回答以下問題（可在下方新增文字儲存格作答）：

### 1. K-means 和 Random Forest 結果的最大差異是什麼？
提示：比較兩種方法的分類圖，觀察空間分布的差異。非監督 vs. 監督方法各有什麼優缺點？

### 2. 哪些類別最容易混淆？為什麼？
提示：參考混淆矩陣，找出 off-diagonal 數值較大的類別組合。思考它們的光譜特徵為什麼相似。

### 3. 如果你是指揮官，你最在乎哪個類別的 accuracy？
提示：在 2024 花蓮地震的災害應變情境中，哪個類別的誤判後果最嚴重？User's Accuracy 和 Producer's Accuracy 哪個更重要？